# 08 · encoder screen on Colab (T4)
Runs `experiments/04-plm-indobert/plm_qa.py` for each encoder and seed, exactly as the Kaggle kernels do
(`experiments/08-plm-encoder-screen/kaggle/`). Each run fine-tunes on train, picks the epoch and the no-answer
threshold on dev, and predicts `official_val`. Finished seeds are copied to `mrc08/runs/` on Drive and skipped on a
rerun, so after a disconnect just run all cells again. Runtime: base models about 7 min per seed, large about 25.

Put `mrc08_bundle.zip` in the same Drive folder as this notebook (`proyek_akhir/mrc08`).


In [ ]:
# Which jobs to run. Remove entries to run fewer; the order is the run order.
JOBS = [
 [
  "04-plm-indobert",
  "plm_indobert_full_r2",
  "indobenchmark/indobert-base-p1",
  "base"
 ],
 [
  "08-plm-encoder-screen",
  "mdeberta_v3_base",
  "microsoft/mdeberta-v3-base",
  "base"
 ],
 [
  "08-plm-encoder-screen",
  "xlmr_large",
  "FacebookAI/xlm-roberta-large",
  "large"
 ],
 [
  "08-plm-encoder-screen",
  "nusabert_large",
  "LazarusNLP/NusaBERT-large",
  "large"
 ],
 [
  "08-plm-encoder-screen",
  "albert_indolite_large",
  "indobenchmark/indobert-lite-large-p2",
  "large"
 ],
 [
  "08-plm-encoder-screen",
  "electra_indo_base",
  "ChristopherA08/IndoELECTRA",
  "base"
 ]
]
SEEDS = [13, 42, 87]
SIZE = {"base": {"lr": "3e-5", "bs": 16, "accum": 1}, "large": {"lr": "1.5e-5", "bs": 8, "accum": 2}}
BASE = {"epochs": 3, "warmup": 0.1}
BUNDLE_SHA256 = "f63c1044a802cbaa64e5b4a85a72e4c51772afa93cf0cf75881c12aaf7daf4c0"


In [ ]:
import glob, hashlib, json, os, shutil, subprocess, sys, time, zipfile
from google.colab import drive
drive.mount("/content/drive")
cands = [p for pat in ("/content/drive/MyDrive/mrc08", "/content/drive/MyDrive/*/mrc08", "/content/drive/MyDrive/*/*/mrc08",
                       "/content/drive/MyDrive/*/*/*/mrc08", "/content/drive/MyDrive/*/*/*/*/mrc08")
         for p in glob.glob(pat) if os.path.exists(p + "/mrc08_bundle.zip")]
assert cands, "folder mrc08 with mrc08_bundle.zip not found under MyDrive (add a shortcut to My Drive if it is shared)"
DRV = cands[0]; print("Drive folder:", DRV)
z = DRV + "/mrc08_bundle.zip"
assert hashlib.sha256(open(z, "rb").read()).hexdigest() == BUNDLE_SHA256, "bundle hash differs: rebuild the notebook"
shutil.rmtree("/content/proj", ignore_errors=True)
zipfile.ZipFile(z).extractall("/content")
os.chdir("/content/proj")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.4.0", "sentencepiece", "protobuf"], check=True)
import torch, transformers
print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))


In [ ]:
status_path = DRV + "/runs/status.json"
os.makedirs(DRV + "/runs", exist_ok=True)
status = json.load(open(status_path)) if os.path.exists(status_path) else {}
todo = [(j, s) for j in JOBS for s in SEEDS]
for n, ((exp, run, model, size), seed) in enumerate(todo, 1):
    dest = f"{DRV}/runs/{exp}/{run}/seed{seed}"
    if os.path.exists(dest + "/metrics_dev.json") and os.path.exists(dest + "/preds_official_val.jsonl"):
        print(f"[{n}/{len(todo)}] skip {run} seed{seed} (on Drive)"); continue
    hp, out = SIZE[size], f"/content/runs/{exp}/{run}/seed{seed}"
    cmd = [sys.executable, "experiments/04-plm-indobert/plm_qa.py", "--device", "cuda", "--fp16", "--run-name", run,
           "--model-name", model, "--seed", str(seed), "--epochs", str(BASE["epochs"]), "--warmup-ratio", str(BASE["warmup"]),
           "--lr", hp["lr"], "--batch-size", str(hp["bs"]), "--grad-accum", str(hp["accum"]), "--out-dir", out,
           "--predict-file", "data/processed/official_val.jsonl"]
    print(f"[{n}/{len(todo)}]", " ".join(cmd[2:]), flush=True)
    t0 = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:  # keep the cell output short: epoch summaries, errors, and every 50th step
        if line.startswith(("==", "best epoch", "wrote", "train examples", "params", "Traceback")) or "Error" in line \
                or (" step " in line and int(line.split(" step ")[1].split("/")[0]) % 50 == 0):
            print("   ", line.rstrip(), flush=True)
    rc = p.wait()
    m = out + "/metrics_dev.json"
    f1 = json.load(open(m))["at_best_threshold"]["f1"] if os.path.exists(m) else None
    status[f"{run}/seed{seed}"] = {"rc": rc, "dev_f1": f1, "minutes": round((time.time() - t0) / 60, 1),
                                   "gpu": torch.cuda.get_device_name(0)}
    print("    STATUS", status[f"{run}/seed{seed}"], flush=True)
    if rc == 0:
        shutil.copytree(out, dest, dirs_exist_ok=True)
    json.dump(status, open(status_path, "w"), indent=1)
print(json.dumps(status, indent=1))


In [ ]:
# Zip everything finished so far (Drive: mrc08/results/), for pulling into experiments/<exp>/runs/ locally
stamp = time.strftime("%Y%m%d-%H%M")
os.makedirs(DRV + "/results", exist_ok=True)
print(shutil.make_archive(f"{DRV}/results/mrc08_runs_{stamp}", "zip", DRV + "/runs"))
